*0.3 Classical NLP*

# Tokenization: subword

**The situation.** A word-level model is trained on 100,000 words. In production a customer writes "chargeback" — not in the vocabulary. The model sees `<unknown>`. So does "unrefundable", every product code, every typo. In one week, 8% of tokens are `<unknown>`, and the model is blind exactly where the customer's problem is.

**Subword tokenization.** Instead of whole words, the vocabulary holds *pieces*: common words stay whole, rare words split into known parts. "chargeback" → "charge" + "back". "unrefundable" → "un" + "refund" + "able". Nothing is unknown; the worst case is one piece per character. Every modern language model tokenizes this way. The next four items are the four main algorithms for choosing the pieces.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)  # a WordPiece vocabulary of 30,522 pieces
words = ["charge", "chargeback", "unrefundable", "SKU-88213", "recieve"]
for word in words:
    pieces = tokenizer.tokenize(word)
    print(f"{word:<14} → {pieces}")
assert tokenizer.unk_token not in tokenizer.tokenize("unrefundable")

charge         → ['charge']
chargeback     → ['charge', '##back']
unrefundable   → ['un', '##re', '##fu', '##nda', '##ble']
SKU-88213      → ['sk', '##u', '-', '88', '##21', '##3']
recieve        → ['rec', '##ie', '##ve']


**Reading the output.** "charge" is one piece. "chargeback" and "unrefundable" break into meaningful parts (`##` marks "continues the previous piece"). The product code and the typo break into small fragments — ugly, but not unknown, so the model still gets a signal.

```
word-level    chargeback  → <unknown>          blind
subword       chargeback  → charge + ##back    knows both parts
```

**The rule to remember.** Subwords make the vocabulary finite and the unknowns zero. Common → one piece; rare → several; that is also why rare words cost more tokens.

| Use it when | Don't when | Instead use |
|---|---|---|
| any neural model — you do not choose, the model's tokenizer comes with it | classical pipelines where whole words are the feature | word tokenization |

**Watch out**
- Token counts drive cost and context limits. Non-English text and code split into more pieces per word — often 2–3× the tokens.
- Never mix tokenizers: a model only understands the pieces it was trained with.
- Numbers split oddly ("1250" → "125" + "0"); arithmetic is hard for models partly because of this.